In [33]:
 # ! If crashes
#python -m ipykernel install --user --name torch_stable --display-name "Python (torch_stable)"
#import os
#os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
#os.environ["OMP_NUM_THREADS"] = "1"

In [34]:
import pandas as pd 

data=pd.read_csv('100_Unique_QA_Dataset.csv')
data.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [35]:
 # TODO: Tokenizing the data

def tokenize(text):
    text=text.lower()
    text=text.replace('?','')
    text=text.replace("'","")
    return text.split()

In [36]:
# Vocab 
vocab={'<UNK>':0}

def build_vocab(row):
    tokenized_question=tokenize(row['question'])
    tokenized_ans=tokenize(row['answer'])

    merged_tokens=tokenized_question + tokenized_ans

    for token in merged_tokens:
        if token not in vocab:
            vocab[token]=len(vocab)



In [37]:
data.apply(build_vocab,axis=1)


0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [38]:
 # Convert words to numerical indices 

def text_to_indices(text,vocab):
    indexed_text=[]

    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])

    return indexed_text
    
 # TODO: text_to_indices('Captial of india?',vocab) # Output [0, 5, 73]

In [39]:
import sys
print(sys.version)


3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]


In [40]:
import torch as tp
print(tp.__version__)

2.9.1+cpu


In [41]:
import torch as tp
from torch.utils.data import DataLoader,Dataset

class QADataset(Dataset):

    def __init__(self,data,vocab):
        self.data=data
        self.vocab=vocab
        
    def __len__(self):
        return self.data.shape[0]
    def __getitem__(self,index):
        num_quest=text_to_indices(self.data.iloc[index]['question'],self.vocab)
        num_ans=text_to_indices(self.data.iloc[index]['answer'],self.vocab)

        return tp.tensor(num_quest),tp.tensor(num_ans)


In [42]:
dataset=QADataset(data,vocab)


In [43]:
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)

In [44]:
for question , answer in dataloader:
    print(question,answer)

tensor([[ 78,  79, 150, 151,  14, 152, 153]]) tensor([[154]])
tensor([[10, 55,  3, 56,  5, 57]]) tensor([[58]])
tensor([[1, 2, 3, 4, 5, 8]]) tensor([[9]])
tensor([[  1,   2,   3,  17, 115,  83,  84]]) tensor([[116]])
tensor([[ 1,  2,  3, 59, 25,  5, 26, 19, 60]]) tensor([[61]])
tensor([[  1,   2,   3,   4,   5, 113]]) tensor([[114]])
tensor([[ 1,  2,  3,  4,  5, 99]]) tensor([[100]])
tensor([[ 42, 137,   2, 226,  12,   3, 227, 228]]) tensor([[155]])
tensor([[ 42,   2,   3, 274, 211, 275]]) tensor([[276]])
tensor([[  1,   2,   3, 221,   5, 222, 223, 224]]) tensor([[225]])
tensor([[  1,   2,   3,   4,   5, 206]]) tensor([[207]])
tensor([[10, 11, 12, 13, 14, 15]]) tensor([[16]])
tensor([[ 42, 101,   2,   3,  17]]) tensor([[102]])
tensor([[  1,   2,   3,  37,  38,  39, 161]]) tensor([[162]])
tensor([[ 1,  2,  3, 92, 93, 94]]) tensor([[95]])
tensor([[ 78,  79, 288,  81,  19,  14, 289]]) tensor([[85]])
tensor([[ 10,  11, 189, 158, 190]]) tensor([[191]])
tensor([[  1,   2,   3,   4,   5, 286]

In [45]:
import torch.nn as nn 
class SimpleRNN(nn.Module):
    pass
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,embedding_dim=50) #! This is embedding Layer
        self.rnn=nn.RNN(50,90,batch_first=True)
        self.fc=nn.Linear(90,vocab_size)

    def forward(self,question):
        embedded_quest=self.embedding(question)
        hidden,final=self.rnn(embedded_quest)
        output=self.fc(final.squeeze(0))

        return output


In [46]:
learn_rate=0.001
epochs=20


In [47]:
model=SimpleRNN(len(vocab))

In [48]:
criterion=nn.CrossEntropyLoss()
optimizer=tp.optim.Adam(model.parameters(),lr=learn_rate)

In [49]:
# Training Loop
for epoch in range(epochs):
    total_loss=0
    for question,answer in dataloader:
        optimizer.zero_grad()
        #! Forward pass
        output=model(question)
        #! loss
        loss=criterion(output,answer[0])
        #! gradient
        loss.backward()
        #! update gradients
        optimizer.step()
        total_loss=total_loss+loss.item()
    print(f'Epochs: {epoch+1}, Loss: {total_loss}')


Epochs: 1, Loss: 523.2973771095276
Epochs: 2, Loss: 437.2613034248352
Epochs: 3, Loss: 343.1186912059784
Epochs: 4, Loss: 269.90622556209564
Epochs: 5, Loss: 207.41599786281586
Epochs: 6, Loss: 149.8429024219513
Epochs: 7, Loss: 104.58146902918816
Epochs: 8, Loss: 73.46493378281593
Epochs: 9, Loss: 52.189651891589165
Epochs: 10, Loss: 37.59602144360542
Epochs: 11, Loss: 28.44006135314703
Epochs: 12, Loss: 22.108140513300896
Epochs: 13, Loss: 17.73433119803667
Epochs: 14, Loss: 14.215238682925701
Epochs: 15, Loss: 11.652783688157797
Epochs: 16, Loss: 9.714740846306086
Epochs: 17, Loss: 8.115972027182579
Epochs: 18, Loss: 7.020246960222721
Epochs: 19, Loss: 5.963446099311113
Epochs: 20, Loss: 5.172181336209178


In [59]:
 #? Prediction
def predict(model,question,threshold=0.3):

    #! convert Q to numbers
    num_quest=text_to_indices(question,vocab)
    #! tensor
    quest_tensor=tp.tensor(num_quest).unsqueeze(0)
    #! send to model
    output=model(quest_tensor)
    
    #print(output.shape)
    #!convert logits to probab
    probab=tp.nn.functional.softmax(output,dim=1)
    
    #! find index of max probab
    value,index=tp.max(probab,dim=1)

    if value<threshold:
        print("I Dont Know")

    print(list(vocab.keys())[index])


In [73]:
predict(model,'what is the capital of india')

delhi
